# 03. Apply predefined data-quality range rules and define the final cohort

This notebook applies the predefined analytic data-quality ranges and creates the
primary analytic sample. The lower bound of each range excludes values that are
not physiologically possible; the upper bound excludes extreme values. Both are
reported, and the next section quantifies how many excluded records met the
criteria for metabolic syndrome.

## Main tasks

- Apply the 12 ordered range rules and record every exclusion.
- Quantify whether the exclusions remove metabolic-syndrome cases selectively.
- Save the final cohort and manuscript summary tables.

The final count is computed from the corrected missing-data and range rules.
The previously reported count (13,670) is printed only as a comparison, not as
an assertion.


In [1]:
# Imports and paths
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Use environment-based paths before publication; local paths are retained here.
BASE = Path(os.environ.get("METS_DATA", r"D:\eda\mets\data"))
OUT  = Path(os.environ.get("METS_OUT",  r"D:\eda\mets\output"))
for _d in (BASE, BASE/"raw", BASE/"work", OUT, OUT/"tables", OUT/"figures", OUT/"logs"):
    _d.mkdir(parents=True, exist_ok=True)

# Outcomes from notebook 01: IDF primary and modified ATP III sensitivity.
LABELS = ["MS", "MS_atp3"]

# This notebook records its own segment of the participant flow. Notebook 04
# joins the segments in order.
FLOW = []

def keep(d, mask, stage):
    """Keep selected rows and record exclusions, including positive cases."""
    mask = mask.reindex(d.index).fillna(False).astype(bool)
    n_before = len(d)
    out = d.loc[mask].copy()
    excluded = d.loc[~mask]

    record = {"step": len(FLOW), "stage": stage,
              "excluded": n_before - len(out), "remaining": len(out)}
    for label in LABELS:
        if label in d.columns:
            record[f"excluded_{label}"] = int(excluded[label].eq(1).sum())
    FLOW.append(record)

    print(f"  - {stage:<53} -{n_before-len(out):>8,}  ->  {len(out):>9,}"
          f"   (MetS {record.get('excluded_MS', 0):>4,})")
    return out

def save_flow():
    """Write this notebook's flow segment, which notebook 04 draws as Fig 1."""
    flow = pd.DataFrame(FLOW)
    flow.to_csv(OUT/"tables"/"flow_03.csv", index=False, encoding="utf-8-sig")
    return flow


In [2]:
df = pd.read_csv(BASE/"work"/"Obesity_Data_proprecessing_Na.csv")
print(f"loaded {len(df):,} rows, {df.shape[1]} cols")


loaded 13,831 rows, 45 cols


## 1. The range rules, applied in order

In [3]:
# Apply ordered analytic data-quality range rules; bounds are exclusive and order determines per-rule counts.
rules = [
    ("BMI 5-60 kg/m2",                 {"bmi": (5, 60)}),
    ("Waist 20-200 cm",                {"waist": (20, 200)}),
    ("Height 110-210 cm",              {"height": (110, 210)}),
    ("Weight 20-250 kg",               {"weight": (20, 250)}),
    ("BP 60-250 / 40-150 mmHg",        {"sysBP": (60, 250), "diaBP": (40, 150)}),
    ("Glucose 40-600 mg/dL",           {"glucose": (40, 600)}),
    ("Total cholesterol 40-600 mg/dL", {"chol": (40, 600)}),
    ("HDL cholesterol 10-150 mg/dL",   {"HDL": (10, 150)}),
    ("Triglycerides 10-500 mg/dL",     {"TG": (10, 500)}),
    ("LDL cholesterol 20-400 mg/dL",   {"LDL": (20, 400)}),
    ("AST 5-300 U/L",                  {"AST": (5, 300)}),
    ("ALT 5-500 U/L",                  {"ALT": (5, 500)}),
]

print("=== Participant flow (continued) ===")
prevalence_before = df["MS"].mean()
log = []

for name, bounds in rules:
    out_of_range = pd.Series(False, index=df.index)
    for col, (low, high) in bounds.items():
        out_of_range = out_of_range | df[col].le(low) | df[col].ge(high)

    removed = df.loc[out_of_range]
    n_removed = int(out_of_range.sum())
    if n_removed:
        mets_share = round(removed["MS"].mean() * 100, 1)
    else:
        mets_share = np.nan

    log.append({
        "rule": name,
        "n_removed": n_removed,
        "n_removed_MetS": int(removed["MS"].eq(1).sum()),
        "MetS_%_among_removed": mets_share,
        "remaining": len(df) - n_removed,
    })
    df = keep(df, ~out_of_range, f"Outside data-quality range: {name}")

outlier_log = pd.DataFrame(log)
outlier_log.to_csv(OUT/"tables"/"outlier_rules.csv", index=False, encoding="utf-8-sig")
print()
print(outlier_log.to_string(index=False))


=== Participant flow (continued) ===
  - Outside data-quality range: BMI 5-60 kg/m2            -       0  ->     13,831   (MetS    0)
  - Outside data-quality range: Waist 20-200 cm           -      12  ->     13,819   (MetS    1)
  - Outside data-quality range: Height 110-210 cm         -       0  ->     13,819   (MetS    0)
  - Outside data-quality range: Weight 20-250 kg          -       0  ->     13,819   (MetS    0)
  - Outside data-quality range: BP 60-250 / 40-150 mmHg   -      19  ->     13,800   (MetS    1)
  - Outside data-quality range: Glucose 40-600 mg/dL      -       3  ->     13,797   (MetS    0)
  - Outside data-quality range: Total cholesterol 40-600 mg/dL -       1  ->     13,796   (MetS    0)
  - Outside data-quality range: HDL cholesterol 10-150 mg/dL -      29  ->     13,767   (MetS    0)
  - Outside data-quality range: Triglycerides 10-500 mg/dL -      33  ->     13,734   (MetS   24)
  - Outside data-quality range: LDL cholesterol 20-400 mg/dL -      16  ->     13

## 2. Do these exclusions remove cases selectively?

In [4]:
# Compare the proportion meeting MetS criteria among excluded records with the
# cohort before the range rules were applied. The upper triglyceride bound acts on
# a component of the definition, so the rules are expected to remove a slightly
# higher proportion of cases than of the cohort as a whole. The Limitations report
# these two figures.
n_removed = int(outlier_log.n_removed.sum())
n_removed_mets = int(outlier_log.n_removed_MetS.sum())
if n_removed:
    proportion_among_removed = n_removed_mets / n_removed
else:
    proportion_among_removed = np.nan

print(f"total removed            : {n_removed:,}")
print(f"  of which MetS-positive : {n_removed_mets:,}"
      f"  ({proportion_among_removed*100:.1f}%)")
print(f"cohort proportion before : {prevalence_before*100:.1f}%")
if n_removed and prevalence_before > 0:
    print(f"ratio                    : "
          f"{proportion_among_removed/prevalence_before:.2f}x")

print("\nThe per-rule breakdown is in outlier_rules.csv.")


total removed            : 161
  of which MetS-positive : 33  (20.5%)
cohort proportion before : 12.7%
ratio                    : 1.61x

The per-rule breakdown is in outlier_rules.csv.


## 3. Final counts and save

In [5]:
# Report the corrected final cohort. The prior manuscript count is shown only
# to flag whether the corrected complete-case rule changes the sample.
PRIOR_REPORTED_N = 13_670
PRIOR_REPORTED_BY_GRADE = {1: 1521, 2: 5413, 3: 6736}

actual_by_grade = {
    int(k): int(v)
    for k, v in df["global_grade"].value_counts().sort_index().items()
}

print(f"rows            : {len(df):,}")
print(f"by school level : {actual_by_grade}")

if len(df) != PRIOR_REPORTED_N or actual_by_grade != PRIOR_REPORTED_BY_GRADE:
    print("\nNOTE: corrected data-cleaning rules changed the cohort relative to the")
    print("previously reported manuscript counts. Re-run notebooks 04 and 05 and")
    print("update all manuscript counts and model results before submission.")
else:
    print("\nCohort count is unchanged from the previously reported manuscript.")

df.to_csv(
    BASE/"work"/"Obesity_Data_proprecessing_OUTLIER.csv",
    index=False, encoding="utf-8-sig"
)
flow = save_flow()
print(f"\nsaved -> Obesity_Data_proprecessing_OUTLIER.csv  ({len(df):,} rows)")


rows            : 13,670
by school level : {1: 1521, 2: 5413, 3: 6736}

Cohort count is unchanged from the previously reported manuscript.

saved -> Obesity_Data_proprecessing_OUTLIER.csv  (13,670 rows)


## 4. Every number the manuscript reports

In [6]:
# Manuscript cohort summaries; high school is split at the age-16 IDF boundary.
TRUE_TOKENS = {"true", "1", "1.0"}
FALSE_TOKENS = {"false", "0", "0.0"}


def component_flags(column):
    """Read a component written as True/False/blank back into met and assessed."""
    raw = df[column].astype(str).str.strip().str.lower()
    unknown = set(raw.unique()) - TRUE_TOKENS - FALSE_TOKENS - {"nan", "", "none"}
    assert not unknown, f"unrecognised values in {column}: {sorted(unknown)}"
    met = raw.isin(TRUE_TOKENS)
    assessed = raw.isin(TRUE_TOKENS | FALSE_TOKENS)
    return met, assessed


SEX_LABEL = {1: "Male", 0: "Female"}
GRADE_LABEL = {1: "Elementary", 2: "Middle", 3: "High"}
d = df.assign(Sex=df["sex"].map(SEX_LABEL),
              Grade=df["global_grade"].map(GRADE_LABEL))
d["Group"] = d["Grade"]
is_high = d["Grade"] == "High"
d.loc[is_high, "Group"] = "High " + d.loc[is_high, "age_band"]
GROUP_ORDER = ["Elementary", "Middle", "High 10-<16", "High >=16"]

print("=" * 66)
print("FINAL NUMBERS FOR THE MANUSCRIPT   (n = %d)" % len(df))
print("=" * 66)

print("\n[1] Prevalence under each definition")
for label, description in [("MS", "IDF (primary)"),
                           ("MS_atp3", "modified ATP III (sensitivity)")]:
    print(f"  {description:<32} {int(df[label].eq(1).sum()):>6,}"
          f"  ({df[label].mean()*100:5.2f}%)")

print("\n[2] Prevalence by age band, the boundary the IDF criteria use")
print(d.groupby("age_band")[["MS", "MS_atp3"]]
       .agg(["count", "sum", "mean"]).round(4).to_string())

print("\n[3] Prevalence by sex")
print(d.groupby("Sex")[["MS", "MS_atp3"]].mean().round(4).to_string())

print("\n[4] Prevalence by sex and group  -> Fig 2")
prevalence = (d.groupby(["Sex", "Group"])[["MS", "MS_atp3"]]
                .agg(["count", "sum", "mean"]).round(4))
print(prevalence.reindex(GROUP_ORDER, level="Group").to_string())
prevalence.to_csv(OUT/"tables"/"prevalence_by_definition.csv",
                  encoding="utf-8-sig")

print("\n[5] Sex composition of the group classified positive")
for label, description in [("MS", "IDF"), ("MS_atp3", "ATP III")]:
    positive = d[d[label] == 1]
    print(f"  {description:<14} male {int((positive.Sex=='Male').sum()):>5,}"
          f" / {len(positive):>5,}"
          f" = {(positive.Sex=='Male').mean()*100:5.2f}%")
print(f"  {'cohort':<14} male {int((d.Sex=='Male').sum()):>5,} / {len(d):>5,}"
      f" = {(d.Sex=='Male').mean()*100:5.2f}%")

# Diagnostic around the age-16 criterion change.
print("\n[6] Prevalence around the 16th birthday")
window = d[(d.age >= 14) & (d.age < 18)].copy()
window["age_bin"] = pd.cut(window["age"], np.arange(14, 18.5, 0.5))
discontinuity = (window.groupby("age_bin", observed=True)[["MS", "MS_atp3"]]
                       .agg(["count", "mean"]))
print(discontinuity.round(4).to_string())

print("\n[7] Baselines for the modelling notebooks (primary outcome MS)")
n_positive = int(df.MS.eq(1).sum())
n = len(df)
print(f"  prevalence                     : {n_positive/n*100:.2f}%")
print(f"  accuracy of predicting nobody  : {1 - n_positive/n:.4f}")
print("    Any reported accuracy has to be read against this value, which is")
print("    obtained at a sensitivity of zero.")
print(f"  expected positives in a 20% test set: "
      f"about {round(n_positive*0.2):,} of {round(n*0.2):,}")

print("\n[8] Central obesity, the obligatory IDF criterion")
meets_central_obesity, _ = component_flags("comp_wc")
print(f"  met by {int(meets_central_obesity.sum()):,} / {n:,}"
      f"  ({meets_central_obesity.mean()*100:.1f}%)")
print(f"  prevalence within that group   : "
      f"{df.loc[meets_central_obesity, 'MS'].mean()*100:.2f}%")
print("    These outcome-defining components are excluded from the model predictors.")

# C7 (Editor 3): the Methods describe the cohort with these figures.
counts = df["obesity_class"].value_counts()
rows = []
for label, n_class in counts.items():
    rows.append({"measure": f"{label}, n", "value": float(n_class)})
    rows.append({"measure": f"{label}, %", "value": round(n_class / len(df) * 100, 2)})
rows.append({"measure": "BMI z-score, mean", "value": round(df["bmi_z"].mean(), 4)})
rows.append({"measure": "BMI z-score, SD", "value": round(df["bmi_z"].std(ddof=1), 4)})
obesity_profile = pd.DataFrame(rows)
print("\n[9] Obesity profile of the final cohort")
print(obesity_profile.to_string(index=False))
obesity_profile.to_csv(OUT/"tables"/"cohort_obesity_profile.csv", index=False,
                       encoding="utf-8-sig")

# Component and definition-agreement tables (Editor 1), on the reported sample.
COMPONENT_LABEL = {
    "wc":  "Central obesity (WC >= cut-off)",
    "bp":  "Elevated BP (>=130/85 mmHg)",
    "hdl": "Reduced HDL cholesterol",
    "tg":  "Elevated triglycerides (>=150 mg/dL)",
    "glu": "Elevated fasting glucose (>=100 mg/dL)",
}
rows = []
for key, label in COMPONENT_LABEL.items():
    met, assessed = component_flags(f"comp_{key}")
    rows.append({"Component": label,
                 "n_assessed": int(assessed.sum()),
                 "n_positive": int(met.sum()),
                 "n_missing": int((~assessed).sum()),
                 "prevalence_%": round(met.sum() / assessed.sum() * 100, 1)})

s1a = pd.DataFrame(rows)
print("\n[10] How often each component was met")
print(s1a.to_string(index=False))
s1a.to_csv(OUT/"tables"/"mets_component_prevalence.csv", index=False,
           encoding="utf-8-sig")

both = df[["MS", "MS_atp3"]].dropna()
n11 = int(((both.MS == 1) & (both.MS_atp3 == 1)).sum())
n10 = int(((both.MS == 1) & (both.MS_atp3 == 0)).sum())
n01 = int(((both.MS == 0) & (both.MS_atp3 == 1)).sum())
n00 = int(((both.MS == 0) & (both.MS_atp3 == 0)).sum())
n_both = len(both)

observed_agreement = (n11 + n00) / n_both
expected_agreement = ((n11 + n10) * (n11 + n01)
                      + (n01 + n00) * (n10 + n00)) / n_both ** 2
kappa = (observed_agreement - expected_agreement) / (1 - expected_agreement)

print("\n[11] IDF against modified ATP III")
print(pd.crosstab(both.MS, both.MS_atp3, margins=True).to_string())
print(f"  discordant {n10 + n01:,} ({(n10 + n01) / n_both * 100:.2f}%)"
      f" | kappa {kappa:.4f}")

# Every IDF case must also satisfy the ATP III count rule.
assert n10 == 0, f"{n10} participants are IDF-positive but ATP III-negative"
print("  every IDF case is also an ATP III case")

pd.DataFrame({"IDF": [n11 + n10], "ATP3": [n11 + n01],
              "discordant": [n10 + n01], "kappa": [round(kappa, 4)]}
             ).to_csv(OUT/"tables"/"definition_agreement.csv",
                      index=False, encoding="utf-8-sig")

# C6 (Editor 1): the HDL band where the two definitions part company.
# ATP III applies the adult female cut-off (<50) at every age; the IDF
# pediatric criteria apply <40 below 16 years. The HDL criterion therefore
# differs only for girls whose value falls between the two. The definitions
# differ in other ways as well - central obesity is obligatory under IDF and
# the count rule is not the same - so the two prevalence figures below are not
# the effect of the threshold on its own.
girls_under_16 = d[(d["age"] < 16) & (d["sex"] == 0)]
in_band = girls_under_16["HDL"].between(40, 49.999)
hdl_band = pd.DataFrame([{
    "group": "Females aged 10 to <16 years",
    "n": len(girls_under_16),
    "n_HDL_40_to_49": int(in_band.sum()),
    "percent_HDL_40_to_49": round(in_band.mean() * 100, 1),
    "MetS_%_in_band_ATP3": round(girls_under_16.loc[in_band, "MS_atp3"].mean() * 100, 2),
    "MetS_%_in_band_IDF": round(girls_under_16.loc[in_band, "MS"].mean() * 100, 2),
}])
print("\n[12] HDL 40-49 mg/dL band, females aged 10 to <16 years")
print(hdl_band.to_string(index=False))
hdl_band.to_csv(OUT/"tables"/"hdl_discordance.csv", index=False,
                encoding="utf-8-sig")
print("  This band is where the definitions disagree on HDL, not the")
print("  proportion with reduced HDL cholesterol under either one. The two")
print("  prevalence figures also carry the obligatory central-obesity")
print("  criterion, so neither isolates the threshold.")


FINAL NUMBERS FOR THE MANUSCRIPT   (n = 13670)

[1] Prevalence under each definition
  IDF (primary)                     1,730  (12.66%)
  modified ATP III (sensitivity)    2,203  (16.12%)

[2] Prevalence by age band, the boundary the IDF criteria use
             MS                 MS_atp3                
          count     sum    mean   count     sum    mean
age_band                                               
10-<16    10928  1286.0  0.1177   10928  1741.0  0.1593
>=16       2742   444.0  0.1619    2742   462.0  0.1685

[3] Prevalence by sex
            MS  MS_atp3
Sex                    
Female  0.1126   0.1927
Male    0.1352   0.1416

[4] Prevalence by sex and group  -> Fig 2
                      MS                MS_atp3               
                   count    sum    mean   count    sum    mean
Sex    Group                                                  
Female Elementary    559   47.0  0.0841     559  111.0  0.1986
       Middle       2042  225.0  0.1102    2042  442.0

---

## Next

Notebook 04 uses the final analytic dataset and completed participant flow to create Table 1 and Figures 1–3.
